In [14]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from matplotlib.backends.backend_pdf import PdfPages

# 1. 修复字体问题：改用系统自带的通用字体
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Bitstream Vera Sans', 'Computer Modern Sans Serif', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False 

# 2. 确保保存的 PDF 中的文本是可编辑的字体 (TrueType)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# 设置绘图风格：保留外边框，加粗边框线
sns.set_theme(style="white", rc={"axes.edgecolor": "#333333", "axes.linewidth": 1.5})

def custom_visualize_to_multipage_pdf(df, dataset_name, target_columns, output_dir='plots'):
    """
    对 DataFrame 中指定的列进行统计并绘图，按字母排序且展示全部数据。
    """
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    columns = [col for col in target_columns if col in df.columns]
    
    if not columns:
        print(f"[{dataset_name}] 没有找到指定的列，跳过绘图。")
        return

    # 填充缺失值
    df = df.fillna('Missing')
    pdf_path = os.path.join(output_dir, f"{dataset_name}_visualizations.pdf")
    
    # 将指定的三列全部强制设为扇形图
    pie_columns = ['main_spatial_layer', 'main_Phenotype_label', 'Phenotype_type']
    
    with PdfPages(pdf_path) as pdf:
        for col in columns:
            # 统计并按 index (即类别名称) 进行字母排序
            counts = df[col].value_counts().sort_index()
            n_unique = len(counts)
            
            if n_unique == 0:
                fig, ax = plt.subplots(figsize=(10, 8))
                ax.text(0.5, 0.5, f"Empty Column: {col}", ha='center', fontsize=14)
                pdf.savefig(fig)
                plt.close()
                continue
            
            # 特殊列：强行使用扇形图
            if col in pie_columns:
                fig, ax = plt.subplots(figsize=(10, 8))
                # 绘制扇形图
                ax.pie(counts.values, labels=counts.index, autopct='%1.1f%%', 
                       startangle=140, colors=sns.color_palette('pastel'))
                ax.set_title(f"{dataset_name}: {col}", fontsize=16, fontweight='bold', pad=20)
            
            # 其他列：使用柱状图
            else:
                # 动态计算画布宽度：基础宽度12，类别多时自动拉长
                fig_width = max(12.0, n_unique * 0.6)
                fig, ax = plt.subplots(figsize=(fig_width, 8))
                
                # 画柱状图，加上 dodge=False 防止柱子变细
                sns.barplot(x=counts.index, y=counts.values, ax=ax, hue=counts.index, palette='Set3', dodge=False)
                
                # 手动移除图例以兼容旧版 seaborn
                if ax.get_legend() is not None:
                    ax.get_legend().remove()
                
                # 在每个柱子上方添加具体的 Count 数值
                for container in ax.containers:
                    ax.bar_label(container, padding=3, color='#5c4040', fontweight='bold', fontsize=10)
                
                ax.set_title(f"{dataset_name}: {col}", fontsize=16, fontweight='bold', pad=20)
                ax.set_ylabel('Count', fontsize=14, fontweight='bold')
                ax.set_xlabel(col, fontsize=14, fontweight='bold')
                
                # 横坐标标签垂直旋转 90 度
                plt.xticks(rotation=90, fontsize=11)
                plt.yticks(fontsize=11)
            
            plt.tight_layout()
            # 保存到 PDF 页面
            pdf.savefig(fig, bbox_inches='tight')
            plt.close() 
            
    print(f"已生成多页 PDF: {pdf_path} (共 {len(columns)} 页)")

# ================= 数据处理 ================= #

target_cols_cell = ['main_cancer_type', 'big_cell_type', 'big__cell_type', 'main_Phenotype_label', 'Phenotype_type']
target_cols_spatial = ['main_cancer_type', 'main_spatial_layer', 'main_Phenotype_label', 'Phenotype_type']

cell_csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv'
spatial_csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv'

try:
    cell_df = pd.read_csv(cell_csv_path)
    spatial_df = pd.read_csv(spatial_csv_path)

    print("正在处理 CellType 数据...")
    custom_visualize_to_multipage_pdf(cell_df, 'CellType', target_columns=target_cols_cell)

    print("\n正在处理 SpatialLayer 数据...")
    custom_visualize_to_multipage_pdf(spatial_df, 'SpatialLayer', target_columns=target_cols_spatial)

    print("\n所有图表已渲染完成。")

except Exception as e:
    print(f"运行出错: {e}")

正在处理 CellType 数据...
已生成多页 PDF: plots/CellType_visualizations.pdf (共 4 页)

正在处理 SpatialLayer 数据...
已生成多页 PDF: plots/SpatialLayer_visualizations.pdf (共 4 页)

所有图表已渲染完成。


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math
import os
#cell_csv_path = '/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv'
cell_csv_path='/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv'
data=pd.read_csv(cell_csv_path)


# 假设你的数据变量名为 data
# 如果你只想统计前几行，可以将 data 替换为 data.head()

# 遍历数据框的每一列
for column in data.columns:
    print(f"--- 列名: {column} ---")
    
    # 1. 统计该列有多少个类别 (种类数量)
    unique_count = data[column].nunique()
    print(f"种类总数: {unique_count}")
    
    # 2. 统计每个类别分别有多少个 (每个类别的频数)
    value_counts = data[column].value_counts()
    print("各类别统计:")
    print(value_counts)
    
    print("\n") # 打印空行方便阅读

--- 列名: SLID ---
种类总数: 75
各类别统计:
SL001    1
SL057    1
SL055    1
SL054    1
SL053    1
        ..
SL025    1
SL024    1
SL023    1
SL022    1
SL075    1
Name: SLID, Length: 75, dtype: int64


--- 列名: species ---
种类总数: 3
各类别统计:
human    65
Human     8
mouse     2
Name: species, dtype: int64


--- 列名: tissue_class ---
种类总数: 28
各类别统计:
liver                      10
brain                       8
Lung                        6
colon                       6
Pancancer                   5
skin                        5
gastric                     3
bladder                     3
lung                        3
skin, lymph nodes (LNs)     2
kidney                      2
Gut                         2
breast                      2
thyroid                     2
gastric，peritoneal          2
Pancreas                    2
esophageal                  1
Tonsils                     1
Thyroid                     1
stomach                     1
skin,lymph                  1
Ovarian                     1
Brain

In [3]:
data.head()

,SLID,species,tissue_class,tissue_type,main_cancer_type,cancer_type,cancer_type_detail,main_spatial_layer,spatial_layer,Cell_type_composition,...,technology_type_for_discovery,technology_platform_for_discovery,Phenotype_type,main_Phenotype_label,Phenotype_label,model_type,technology_type_for_validation,technology_platform_for_validation,evidence_type,Phenotype_evidence
0,SL001,human,colon,colon,Colorectal cancer,Colorectal cancer,CRC,Cell co-localization,colocalised adenoma and cancer epithelial cell...,"Epithelial tumour cells, Tregs",...,"Spatial transcriptomics, scRNA-seq data","10× Genomics, 10× Genomics",Biological phenotype,Immune Tolerance,induce immune tolerance,clinical samples,immunofluorescence,unknow,protein,"Thus, our results suggested that Treg-colocali..."
1,SL002,human,bladder,bladder,Bladder cancer,Bladder cancer,bladder cancer,Cellular neighbourhoods,C1 (epithelial and basal cells),"Epithelial cells, Basal cells",...,Spatial transcriptome,10× Genomics,Biological phenotype,Tumor Recurrence,Tumor Recurrence,clinical samples,multiplex immunofluorescence,Akoya,protein,"Interestingly, spatially proximal differential..."
2,SL003,human,bladder,bladder,Bladder cancer,Bladder cancer,bladder cancer,Cellular neighbourhoods,C0 (NK and T cells),"NK cells, T cells",...,Spatial transcriptome,10× Genomics,Biological phenotype,Tumor Recurrence,Tumor Recurrence,clinical samples,multiplex immunofluorescence,Akoya,protein,"Interestingly, spatially proximal differential..."
3,SL004,human,bladder,bladder,Bladder cancer,Bladder cancer,Bladder Ewing sarcoma/primitive neuroectoderma...,Cell co-localization,interaction between bladder ES-Epi and bladder...,"Bladder ES-Mast cells, bladder ES-Epi cells",...,"Single-cell RNA-sequencing, spatial transcript...","10X Genomics, 10X Genomics",Biological phenotype,Tumor Invasion,Invasion,clinical samples,"multiple immunoflfluorescence, Functional expe...",unknow,protein,The investigation identified the ‘‘TNFSF12-TNF...
4,SL005,Human,Brain,Brain,Brain cancer,Glioblastoma,Glioblastoma,Spatial niche,Peri-necrotic regions,Hypoxia-TAM(defined as ADM+/Galectin-3+ Mo-TAM...,...,Spatial transcriptome,10× Genomics,Clinical phenotype,Drug Response,Dampen drug delivery,"clinical samples, mouse model","Intravital imaging, Multiplex immunostaining",Zeiss 780 multiphoton microscope\nOpal Polaris...,"protein,mouse",Genetic ablation or pharmacological blockade o...


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 环境配置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="whitegrid")

def visualize_as_requested(df, columns, dataset_name):
    """
    根据要求绘制统计图：Count 为纵坐标，全量展示
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            print(f"列 {col} 不存在于 {dataset_name}")
            continue
            
        # 填充缺失值
        series = df[col].fillna('Unknown')
        
        # 排序处理
        if col == 'year':
            # 年份按数值/时间排序
            series = series.astype(str).str.replace('.0', '', regex=False)
            counts = series.value_counts().sort_index()
        else:
            # 其他分类按频数从高到低排序
            counts = series.value_counts()
            
        n_unique = len(counts)
        if n_unique == 0: continue

        # 2. 绘图类型判断
        if n_unique <= 6 and col != 'year':
            # 类别极少用扇形图
            plt.figure(figsize=(10, 8))
            plt.pie(counts.values, labels=counts.index, autopct='%1.1f%%', 
                    startangle=140, colors=sns.color_palette('pastel'))
            plt.title(f"{dataset_name}: {col} 分布")
        else:
            # 类别较多用柱状图，Count 为纵轴 (y)
            # 根据唯一值数量动态设置宽度，防止横轴拥挤
            width = max(12, n_unique * 0.4)
            plt.figure(figsize=(min(60, width), 8)) # 最大宽度限制在 60
            
            x_labels = [str(x)[:50] for x in counts.index] # 截断过长标签
            
            # 绘制柱状图：x轴为分类，y轴为Count
            ax = sns.barplot(x=x_labels, y=counts.values, hue=x_labels, palette='viridis', dodge=False)
            
            # 移除图例（分类信息已在x轴展示）
            if ax.get_legend():
                ax.get_legend().remove()
            
            # x轴标签旋转90度
            plt.xticks(rotation=90)
            plt.title(f"{dataset_name}: {col} (全量统计, 共 {n_unique} 类)")
            plt.ylabel('数量 (Count)')
            plt.xlabel(col)

        plt.tight_layout()
        save_name = f"{dataset_name}_v_{col}.png"
        plt.savefig(save_name, dpi=100)
        print(f"已生成: {save_name}")
        plt.close()

# 3. 加载数据
cell_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv')
spatial_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv')

# 4. 执行绘图任务
cell_cols = ['main_cancer_type', 'big_cell_type', 'year', 'Phenotype_type', 'main_Phenotype_label']
spatial_cols = ['main_cancer_type', 'main_spatial_layer', 'year', 'Phenotype_type', 'main_Phenotype_label']

print("正在生成 CellType 图表...")
visualize_as_requested(cell_df, cell_cols, 'CellType')

print("\n正在生成 SpatialLayer 图表...")
visualize_as_requested(spatial_df, spatial_cols, 'SpatialLayer')

print("\n统计可视化完成。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 1. 环境与配色配置
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white") # 使用简洁白底

def visualize_pro(df, columns, dataset_name, pie_cols):
    """
    专业可视化函数：支持指定扇形图/柱状图，添加数值标注
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            continue
            
        series = df[col].fillna('Unknown')
        
        # 数据排序逻辑
        if col == 'year':
            series = series.astype(str).str.replace('.0', '', regex=False)
            counts = series.value_counts().sort_index()
        else:
            counts = series.value_counts()
            
        n_unique = len(counts)
        if n_unique == 0: continue

        # --- 绘图逻辑 ---
        
        # 判断是画扇形图还是柱状图
        if col in pie_cols:
            # A. 扇形图 (含百分比)
            plt.figure(figsize=(12, 9))
            # 使用 Set3 或 husl 配色，看起来更高级
            colors = sns.color_palette("Set3", n_colors=n_unique)
            wedges, texts, autotexts = plt.pie(
                counts.values, 
                labels=counts.index, 
                autopct='%1.1f%%', 
                startangle=140, 
                colors=colors,
                pctdistance=0.85, # 百分比距离圆心的距离
                explode=[0.05] * n_unique if n_unique < 10 else None # 少量分类时轻微裂开
            )
            # 设置字体颜色为深色以便阅读
            plt.setp(autotexts, size=10, weight="bold")
            plt.title(f"{dataset_name}: {col} 占比统计", fontsize=16, pad=20, fontweight='bold')
            
        else:
            # B. 柱状图 (Count 为纵坐标，上方带数字)
            # 根据分类数量动态计算宽度
            fig_width = max(12, n_unique * 0.5)
            plt.figure(figsize=(min(80, fig_width), 10))
            
            # 使用渐变色系
            colors = sns.color_palette("husl", n_colors=n_unique)
            ax = sns.barplot(x=counts.index.map(str), y=counts.values, hue=counts.index.map(str), palette=colors, dodge=False)
            
            # 添加数字标注 (在柱子顶部显示 Count)
            for p in ax.patches:
                ax.annotate(f'{int(p.get_height())}', 
                            (p.get_x() + p.get_width() / 2., p.get_height()), 
                            ha = 'center', va = 'center', 
                            xytext = (0, 9), 
                            textcoords = 'offset points',
                            fontsize=10, fontweight='bold')

            if ax.get_legend(): ax.get_legend().remove()
            
            plt.xticks(rotation=90, fontsize=10)
            plt.yticks(fontsize=12)
            plt.title(f"{dataset_name}: {col} 全量分布 (共 {n_unique} 类)", fontsize=18, pad=30, fontweight='bold')
            plt.ylabel('数量 (Count)', fontsize=14)
            plt.xlabel(col, fontsize=14)

        plt.tight_layout()
        save_name = f"{dataset_name}_Final_{col}.png"
        plt.savefig(save_name, dpi=120)
        print(f"成功生成高清图表: {save_name}")
        plt.close()

# 2. 加载数据
cell_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv')
spatial_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv')

# 3. 配置列清单
# CellType 表格要求的列
cell_cols = ['main_cancer_type', 'big_cell_type', 'year', 'Phenotype_type', 'Phenotype_label']
cell_pie_cols = ['year', 'Phenotype_label'] # 您要求的扇形图列

# SpatialLayer 表格要求的列
spatial_cols = ['main_cancer_type', 'main_spatial_layer', 'year', 'Phenotype_type', 'main_Phenotype_label']
spatial_pie_cols = ['year', 'main_Phenotype_label'] # 您要求的扇形图列

# 4. 执行绘图
print("开始生成高级可视化图表...")
visualize_pro(cell_df, cell_cols, 'CellType', cell_pie_cols)
visualize_pro(spatial_df, spatial_cols, 'SpatialLayer', spatial_pie_cols)

print("\n所有操作已完成！请检查当前目录下的图片文件。")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# 1. 路径与环境配置
output_dir = '/mnt/data/ljj/Project_TiPhD/111/plots'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 设置学术配色与字体 (解决 Arial 缺失警告)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style="white")

def visualize_pro_to_pdf(df, columns, dataset_name, pie_cols, w=14, h=8):
    """
    全量统计并保存为 PDF
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            continue
            
        # 填充缺失值并预处理
        series = df[col].fillna('Unknown')
        
        # 排序：年份按时间，其余按频数
        if col == 'year':
            series = series.astype(str).str.replace('.0', '', regex=False)
            counts = series.value_counts().sort_index()
        else:
            counts = series.value_counts()
            
        n_unique = len(counts)
        if n_unique == 0: continue

        # 动态计算画布宽度（柱状图如果类别极多，则拉长宽度）
        current_w = max(w, n_unique * 0.45) if col not in pie_cols else w
        plt.figure(figsize=(current_w, h))

        # --- 绘图逻辑 ---
        if col in pie_cols:
            # A. 扇形图 (强制显示百分比)
            # 如果分类超过15个，为了美观将极小项归为 Others
            plot_counts = counts.copy()
            if n_unique > 15:
                others_sum = plot_counts[15:].sum()
                plot_counts = plot_counts[:15]
                plot_counts['Others'] = others_sum
            
            colors = sns.color_palette("Set3", n_colors=len(plot_counts))
            wedges, texts, autotexts = plt.pie(
                plot_counts.values, 
                labels=plot_counts.index, 
                autopct='%1.1f%%', 
                startangle=140, 
                colors=colors,
                pctdistance=0.85
            )
            plt.setp(autotexts, size=9, weight="bold")
            plt.title(f"{dataset_name}: {col} 分布占比", fontsize=16, fontweight='bold', pad=20)
        else:
            # B. 柱状图 (Count 为纵轴，标注数值)
            colors = sns.color_palette("husl", n_colors=n_unique)
            x_labels = [str(x)[:60] for x in counts.index] # 截断超长标签
            ax = sns.barplot(x=x_labels, y=counts.values, hue=x_labels, palette=colors, dodge=False)
            
            # 添加数字标注 (修复：增加对 NaN 和 0 的检查)
            for p in ax.patches:
                val = p.get_height()
                if np.isfinite(val) and val > 0:
                    ax.annotate(f'{int(val)}', 
                                (p.get_x() + p.get_width() / 2., val), 
                                ha='center', va='bottom', 
                                xytext=(0, 5), 
                                textcoords='offset points',
                                fontsize=9, fontweight='bold')
            
            if ax.get_legend(): ax.get_legend().remove()
            plt.xticks(rotation=90, fontsize=10)
            plt.ylabel('数量 (Count)', fontsize=12)
            plt.title(f"{dataset_name}: {col} 全量统计 (共 {n_unique} 类)", fontsize=16, fontweight='bold', pad=20)

        plt.tight_layout()
        save_path = os.path.join(output_dir, f"{dataset_name}_{col}.pdf")
        plt.savefig(save_path, format='pdf', dpi=300)
        plt.close()
        print(f"已生成 PDF: {save_path}")

# 2. 加载数据
cell_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv')
spatial_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv')

# 3. 指定列与任务
# CellType 任务
cell_cols = ['main_cancer_type', 'big_cell_type', 'year', 'Phenotype_type', 'main_Phenotype_label']
visualize_pro_to_pdf(cell_df, cell_cols, 'CellType', ['year','Phenotype_type', 'main_Phenotype_label'])

# SpatialLayer 任务
spatial_cols = ['main_cancer_type', 'main_spatial_layer', 'year', 'Phenotype_type', 'main_Phenotype_label']
visualize_pro_to_pdf(spatial_df, spatial_cols, 'SpatialLayer', ['year','Phenotype_type','main_spatial_layer', 'main_Phenotype_label'])

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# 1. 路径配置
output_dir = '/mnt/data/ljj/Project_TiPhD/111/plots'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. 基础样式与矢量图字体设置
# 使用 DejaVu Sans 是为了最好的跨平台矢量兼容性
plt.rcParams['font.sans-serif'] = ['DejaVu Sans'] 
plt.rcParams['axes.unicode_minus'] = False
# --- 关键修改：设置 PDF 导出为 TrueType 字体 (Type 42) ---
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

sns.set_theme(style="white")

def visualize_pro_to_pdf(df, columns, dataset_name, pie_cols, w=14, h=10):
    """
    全量统计、字母排序、Set3 配色、矢量 PDF 保存
    """
    df = df.copy()
    
    for col in columns:
        if col not in df.columns:
            print(f"跳过：列 '{col}' 未找到")
            continue
            
        # 数据清理：确保所有标签都是字符串，并处理缺失值
        series = df[col].fillna('Unknown').astype(str)
        
        # --- 核心：按英文字母(Index)排序 ---
        if col == 'year':
            series = series.str.replace('.0', '', regex=False)
            counts = series.value_counts().sort_index()
        else:
            counts = series.value_counts().sort_index()
            
        n_unique = len(counts)
        if n_unique == 0: continue

        # 动态调整宽度
        current_w = max(w, n_unique * 0.5) if col not in pie_cols else w
        plt.figure(figsize=(current_w, h))

        # --- 绘图逻辑 ---
        if col in pie_cols:
            # A. 扇形图
            colors = sns.color_palette("Set3", n_colors=n_unique)
            wedges, texts, autotexts = plt.pie(
                counts.values, 
                labels=counts.index, 
                autopct='%1.1f%%', 
                startangle=140, 
                colors=colors,
                pctdistance=0.85
            )
            plt.setp(autotexts, size=9, weight="bold")
            plt.title(f"{dataset_name}: {col}", fontsize=18, fontweight='bold', pad=25)
        else:
            # B. 柱状图
            colors = sns.color_palette("Set3", n_colors=n_unique)
            # 确保标签不包含会导致渲染错误的特殊符号，截断过长标签
            x_labels = [str(x)[:60] for x in counts.index]
            
            ax = sns.barplot(x=x_labels, y=counts.values, hue=x_labels, palette=colors, dodge=False)
            
            # 标注数字
            for p in ax.patches:
                val = p.get_height()
                if np.isfinite(val) and val > 0:
                    ax.annotate(f'{int(val)}', 
                                (p.get_x() + p.get_width() / 2., val), 
                                ha='center', va='bottom', 
                                xytext=(0, 5), 
                                textcoords='offset points',
                                fontsize=10, fontweight='bold')
            
            if ax.get_legend(): ax.get_legend().remove()
                
            plt.xticks(rotation=90, fontsize=11)
            # 修改点：统一使用英文标签，避免中文在矢量图中变成符号
            plt.ylabel('Count', fontsize=14)
            plt.xlabel(col, fontsize=14)
            plt.title(f"{dataset_name}: {col}", fontsize=18, fontweight='bold', pad=25)

        plt.tight_layout()
        
        # 保存设置
        save_path = os.path.join(output_dir, f"{dataset_name}_{col}.pdf")
        # bbox_inches='tight' 确保所有文字都在画布内
        plt.savefig(save_path, format='pdf', dpi=300, bbox_inches='tight')
        print(f"成功保存矢量 PDF: {save_path}")
        plt.close()

# 3. 加载数据 (保持不变)
cell_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/celltype.csv')
spatial_df = pd.read_csv('/mnt/data/ljj/Project_TiPhD/111/data/spatialayer.csv')

# 4. 执行任务 (保持不变)
cell_target_cols = ['main_cancer_type', 'big_cell_type', 'year', 'Phenotype_type', 'main_Phenotype_label']
cell_pies = ['year', 'Phenotype_type', 'main_Phenotype_label']

spatial_target_cols = ['main_cancer_type', 'main_spatial_layer', 'year', 'Phenotype_type', 'main_Phenotype_label']
spatial_pies = ['year', 'Phenotype_type', 'main_spatial_layer', 'main_Phenotype_label']

print("开始生成统计图表...")
visualize_pro_to_pdf(cell_df, cell_target_cols, 'CellType', cell_pies)
visualize_pro_to_pdf(spatial_df, spatial_target_cols, 'SpatialLayer', spatial_pies)

print("\n所有统计分析任务已完成，矢量 PDF 文件已保存。")

开始生成统计图表...
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/CellType_main_cancer_type.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/CellType_big_cell_type.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/CellType_year.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/CellType_Phenotype_type.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/CellType_main_Phenotype_label.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/SpatialLayer_main_cancer_type.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/SpatialLayer_main_spatial_layer.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/SpatialLayer_year.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/SpatialLayer_Phenotype_type.pdf
成功保存矢量 PDF: /mnt/data/ljj/Project_TiPhD/111/plots/SpatialLayer_main_Phenotype_label.pdf

所有统计分析任务已完成，矢量 PDF 文件已保存。
